In [ ]:
import dspy
from dspy.utils import download

In [ ]:
# Uncomment for downloading the necessary json files.
# download("https://huggingface.co/dspy/cache/blob/main/ragqa_arena_tech_corpus.jsonl")
# download("https://huggingface.co/dspy/cache/blob/main/ragqa_arena_tech_examples.jsonl")

In [ ]:
# #connect to LM from your DSPy code
# options = {"num_ctx": 1024}
# lm = dspy.LM('ollama_chat/deepseek-r1:7b', api_base='http://localhost:11434', num_ctx= 8192*2)
# dspy.configure(lm=lm)

In [ ]:
#connect to LM from your DSPy code
lm = dspy.LM('ollama_chat/gpt-oss:20b', api_base='http://localhost:11434')
dspy.configure(lm=lm)

In [ ]:
#Calling LM directly
lm("Say this is a test!", temperature=0.7)  # => ['This is a test!']
lm(messages=[{"role": "user", "content": "Say this is a test in Turkish three times."}])  # => ['This is a test!']

In [ ]:
#Calling LM directly
lm("Say this is a test!", temperature=0.7)  # => ['This is a test!']
lm(messages=[{"role": "user", "content": "Say this is a test in German three times."}])  # => ['This is a test!']

# Basic DSPy Modules

# Math

In [ ]:
math = dspy.ChainOfThought("question -> answer:float")
output = math(question="Two dice are tossed. What is the probability that the sum equals two?Explain to me in detail")
print(output)

In [ ]:
lm_deepseek = dspy.LM('ollama_chat/deepseek-r1:14b', api_base='http://localhost:11434')
with dspy.settings.context(lm=lm_deepseek):
    math = dspy.ChainOfThought("question -> answer:float")
    output = math(question="Two dice are tossed. What is the probability that the sum equals two?Explain to me in detail")
    print(output)
    

In [ ]:
# Converting the fraction to a float programmatically:
import fractions

math = dspy.ChainOfThought("question -> answer")
output = math(question="Two dice are tossed. What is the probability that the sum equals two")
print(output)

# Convert fraction to float
try:
    probability = float(output.answer)
    print(probability)  # Expected: 0.027777...
except ValueError:
    print("Could not convert output to float:", output.answer)


In [ ]:
lm_deepseek = dspy.LM('ollama_chat/deepseek-r1:14b', api_base='http://localhost:11434')
with dspy.settings.context(lm=lm_deepseek):
    math = dspy.ChainOfThought("question -> answer")
    output = math(question="Two dice are tossed. What is the probability that the sum equals two")
    print(output)
    # Convert fraction to float
    try:
        probability = float(output.answer)
        print(probability)  # Expected: 0.027777...
    except ValueError:
        print("Could not convert output to float:", output.answer)

In [ ]:
math = dspy.ChainOfThought("question -> reasoning:str, answer:float")

output = math(question="Two dice are tossed. What is the probability that the sum equals two? ")

print(output)  # Should return a structured output


In [ ]:
output.answer

In [ ]:
qa = dspy.Predict('question: str -> response: str')
response = qa(question="what are high memory and low memory on linux?")

print(response.response)

In [ ]:
dspy.inspect_history(n=1)

In [ ]:
lm_deepseek_14b = dspy.LM('ollama_chat/deepseek-r1:14b', api_base='http://localhost:11434')
with dspy.settings.context(lm=lm_deepseek_14b):
    qa = dspy.Predict('question: str -> response: str')
    response = qa(question="what are high memory and low memory on linux?")

    print(response.response)

In [ ]:

lm_deepseek_8b = dspy.LM('ollama_chat/deepseek-r1:8b', api_base='http://localhost:11434')
with dspy.settings.context(lm=lm_deepseek_8b):
    qa = dspy.Predict('question: str -> response: str')
    response = qa(question="what are high memory and low memory on linux?")

    print(response.response)

In [ ]:
dspy.inspect_history(n=1)

In [ ]:
cot = dspy.ChainOfThought('question -> response')
cot(question="should curly braces appear on their own line?")

## Evaluation

In [ ]:
import ujson
with open("data/ragqa_arena_tech_examples.jsonl") as f:
    data = [ujson.loads(line) for line in f]

In [ ]:
data = [dspy.Example(**d).with_inputs('question') for d in data]

# Let's pick an `example` here from the data.
example = data[2]
example

In [ ]:
import random

random.Random(0).shuffle(data)
trainset, devset, testset = data[:200], data[200:500], data[500:1000]

len(trainset), len(devset), len(testset)

In [ ]:
from dspy.evaluate import SemanticF1

# Instantiate the metric.
metric = SemanticF1(decompositional=True)

# Produce a prediction from our `cot` module, using the `example` above as input.
pred = cot(**example.inputs())

# Compute the metric score for the prediction.
score = metric(example, pred)

print(f"Question: \t {example.question}\n")
print(f"Gold Response: \t {example.response}\n")
print(f"Predicted Response: \t {pred.response}\n")
print(f"Semantic F1 Score: {score:.2f}")

In [ ]:
dspy.inspect_history(n=1)

In [ ]:
# Define an evaluator that we can re-use.
evaluate = dspy.Evaluate(devset=devset, metric=metric, num_threads=24,
                            display_progress=True, display_table=2)

# Evaluate the Chain-of-Thought program.
evaluate(cot)

# RAG

## Basic Retrieval-Augmented Generation (RAG)

In [ ]:
# %conda install -c rapidsai -c conda-forge -c nvidia libcuvs=24.12 'cuda-version>=12.0,<=12.5'

In [ ]:
import ujson
import dspy
from sentence_transformers import SentenceTransformer

In [ ]:
#connect to LM from your DSPy code
lm = dspy.LM('ollama_chat/gpt-oss:20b', api_base='http://localhost:11434')
dspy.configure(lm=lm)

In [ ]:
max_characters = 6000  # for truncating >99th percentile of documents
topk_docs_to_retrieve = 10  # number of documents to retrieve per search query

with open("data/ragqa_arena_tech_corpus.jsonl") as f:
    corpus = [ujson.loads(line)['text'][:max_characters] for line in f]
    print(f"Loaded {len(corpus)} documents. Will encode them below.")


# Load an extremely efficient local model for retrieval
model = SentenceTransformer("sentence-transformers/static-retrieval-mrl-en-v1", device="cuda")
embedder = dspy.Embedder(model.encode)

search = dspy.retrievers.Embeddings(embedder=embedder, corpus=corpus, k=topk_docs_to_retrieve)

In [ ]:
class RAG(dspy.Module):
    def __init__(self):
        self.respond = dspy.ChainOfThought('context, question -> response')

    def forward(self, question):
        context = search(question).passages
        return self.respond(context=context, question=question)

In [ ]:
rag = RAG()
rag(question="Is Emin a friend of Mert? ")

In [ ]:
rag = RAG()
rag(question="what is the title of the paper that received a feedback?")

In [ ]:
dspy.inspect_history()

In [ ]:

lm_deepseek_8b = dspy.LM('ollama_chat/deepseek-r1:8b', api_base='http://localhost:11434')
with dspy.settings.context(lm=lm_deepseek_8b):
    max_characters = 6000  # for truncating >99th percentile of documents
    topk_docs_to_retrieve = 5  # number of documents to retrieve per search query

    with open("data/ragqa_arena_tech_corpus.jsonl") as f:
        corpus = [ujson.loads(line)['text'][:max_characters] for line in f]
        print(f"Loaded {len(corpus)} documents. Will encode them below.")


    # Load an extremely efficient local model for retrieval
    model = SentenceTransformer("sentence-transformers/static-retrieval-mrl-en-v1", device="cuda")
    embedder = dspy.Embedder(model.encode)

    search = dspy.retrievers.Embeddings(embedder=embedder, corpus=corpus, k=topk_docs_to_retrieve)
    rag = RAG()
    rag(question="Is Emin a friend of Mert? ")

In [ ]:
dspy.inspect_history()

In [ ]:
rag = RAG()
rag(question="What is MVC? ")

In [ ]:
dspy.inspect_history()

In [ ]:
with open("data/ragqa_arena_tech_examples.jsonl") as f:
    data = [ujson.loads(line) for line in f]

In [ ]:
data = [dspy.Example(**d).with_inputs('question') for d in data]

# Let's pick an `example` here from the data.
example = data[2]
example

In [ ]:
import random

random.Random(0).shuffle(data)
trainset, devset, testset = data[:200], data[200:500], data[500:1000]

len(trainset), len(devset), len(testset)

In [ ]:
from dspy.evaluate import SemanticF1

# Instantiate the metric.
metric = SemanticF1(decompositional=True)

In [ ]:
# Define an evaluator that we can re-use.
evaluate = dspy.Evaluate(devset=devset, metric=metric, num_threads=24,
                         display_progress=True, display_table=2)

# Evaluate the Chain-of-Thought program.
evaluate(RAG())

## RAG with ColBERTv2 ( Wikipedia Searcher Agent)

In [ ]:
def search_wikipedia(query: str) -> list[str]:
    results = dspy.ColBERTv2(url='https://huggingface.co/dspy/cache/blob/main/wiki.abstracts.2017.tar.gz')(query, k=3)
    return [x['text'] for x in results]

rag = dspy.ChainOfThought('context, question -> response')

question = "What's the name of the castle that David Gregory inherited?"
rag(context=search_wikipedia(question), question=question)

In [ ]:
import wikipediaapi

def search_wikipedia(query: str) -> str:
    wiki_wiki = wikipediaapi.Wikipedia(
        language='en',
        user_agent="LLM-Assistant/1.0 (ecagatay93@email.com)"
    )
    page = wiki_wiki.page(query)

    if not page.exists():
        return "No relevant Wikipedia page found."

    return page.text[:10000]  # Limit content for processing

# Example usage
# question = "For which artist did Kendrick Lamar write the diss track 'Not Like Us'?"
context = search_wikipedia("2024 United States presidential election")

In [ ]:
rag = dspy.ChainOfThought('context, question -> response')

question = "What is the name of running mate of Donald Trump in 2024?"
rag(context=context, question=question)

In [ ]:
rag = dspy.ChainOfThought('context, question -> response')

question = "Did trump claim that the 2020 elections were rigged?"
rag(context=context, question=question)

## Using a DSPy Optimizer to improve your RAG prompt.

In [ ]:
import dspy

In [ ]:
lm = dspy.LM('ollama_chat/deepseek-r1:7b', api_base='http://localhost:11434', num_ctx= 8192*2)
dspy.configure(lm=lm)

In [ ]:
tp = dspy.MIPROv2(metric=metric, auto="medium", num_threads=24)  # use fewer threads if your rate limit is small

optimized_rag = tp.compile(RAG(), trainset=trainset,
                            max_bootstrapped_demos=2, max_labeled_demos=2,
                            requires_permission_to_run=False)

In [ ]:
baseline = rag(question="cmd+tab does not work on hidden or minimized windows")
print(baseline.response)

In [ ]:
pred = optimized_rag(question="cmd+tab does not work on hidden or minimized windows")
print(pred.response)

In [ ]:
evaluate(optimized_rag)

In [ ]:
cost = sum([x['cost'] for x in lm.history if x['cost'] is not None])  # in USD, as calculated by LiteLLM for certain providers

In [ ]:
optimized_rag.save("optimized_rag.json")

loaded_rag = RAG()
loaded_rag.load("optimized_rag.json")

loaded_rag(question="cmd+tab does not work on hidden or minimized windows")

# Classification

In [ ]:
from typing import Literal

class Classify(dspy.Signature):
    """Classify sentiment of a given sentence."""

    sentence: str = dspy.InputField()
    sentiment: Literal['positive', 'negative', 'neutral'] = dspy.OutputField()
    confidence: float = dspy.OutputField()

classify = dspy.Predict(Classify)
classify(sentence="This book was super fun to read")

# Information Extraction

In [ ]:
class ExtractInfo(dspy.Signature):
    """Extract structured information from text."""

    text: str = dspy.InputField()
    title: str = dspy.OutputField()
    headings: str = dspy.OutputField(desc="a JSON string representing a list of headings")
    entities: str = dspy.OutputField(desc="a JSON string representing a list of entities and their metadata")


In [ ]:
import json

module = dspy.Predict(ExtractInfo)
text = "Apple Inc. announced its latest iPhone 14 today." \
    "The CEO, Tim Cook, highlighted its new features in a press release."
response = module(text=text)

print("Raw response:", repr(response))
print(response.title)
print(response.headings)
print(response.entities)


In [ ]:
class ExtractInfo(dspy.Signature):
    """Extract structured information from text."""

    text: str = dspy.InputField()
    title: str = dspy.OutputField()
    headings: list[str] = dspy.OutputField()
    entities: list[dict[str, str]] = dspy.OutputField(desc="a list of entities and their metadata")

In [ ]:
import json

module = dspy.Predict(ExtractInfo)
text = "securite securite all stations this is United States Coast Guard Sector Long Island Sound break the coast guard recieved a report of a vessel that is submerged in the vicinity of Sag Harbor entrance in broad position four one tag zero one decimal eight two north zero seven two tag one eight decimal zero zero west all mariners are requested caution when transiting the area break this is United States Coast Guard Sector Long Island Sound out"
response = module(text=text)

print("Raw response:", repr(response))
print(response.title)
print(response.headings)
print(response.entities)

In [ ]:
class ExtractInfo(dspy.Signature):
    """Extract structured information from text."""

    text: str = dspy.InputField()
    title: str = dspy.OutputField()
    headings: str = dspy.OutputField(desc="a JSON string representing a list of headings")
    entities: str = dspy.OutputField(desc="a JSON string representing a list of entities and their metadata")

In [ ]:
import json

module = dspy.Predict(ExtractInfo)
text = "Kotka VTS, Onki, hyvää iltapäivää. Vessel Strandvej, this is Tavasland."
response = module(text=text)

print("Raw response:", repr(response))
print(response.title)
print(response.headings)
print(response.entities)

# Agents

In [ ]:
import wikipediaapi

def search_wikipedia(query: str) -> str:
    wiki_wiki = wikipediaapi.Wikipedia(
        language='en',
        user_agent="LLM-Assistant/1.0 (ecagatay93@email.com)"
    )
    page = wiki_wiki.page(query)

    if not page.exists():
        return "No relevant Wikipedia page found."

    return page.text  # Limit content for processing

# Example usage
# question = "For which artist did Kendrick Lamar write the diss track 'Not Like Us'?"
context = search_wikipedia("David Gregory (physician)")


In [ ]:
def evaluate_math(expression: str):
    return dspy.PythonInterpreter({}).execute(expression)

def search_wikipedia(query: str) -> str:
    wiki_wiki = wikipediaapi.Wikipedia(
        language='en',
        user_agent="LLM-Assistant/1.0 (ecagatay93@email.com)"
    )
    page = wiki_wiki.page(query)

    if not page.exists():
        return "No relevant Wikipedia page found."

    return page.text[:10000]  # Limit content for processing

react = dspy.ReAct("question -> answer: float", tools=[evaluate_math, search_wikipedia])

pred = react(question="What is 9362158 divided by the year of birth of David Gregory of Kinnairdy castle? Check the wikipedia page for David Gregory (physician)")
print(pred.answer)

In [ ]:
dspy.inspect_history()

# Multi-Stage Pipelines